# Hamsokhan (هم‌سخن) — Autonomous Kaggle Dataset Preprocessing & Sharding
> **Purpose:** Run this standalone notebook to stream, process, and shard Persian datasets (Thomcles, PersianVox, ManaTTS) into contiguous zero-copy `.safetensors` files.  
> **Output:** Automatically creates pre-packaged shards in `/kaggle/working/hamsokhan_shards` which can be published as a **Kaggle Dataset** in 1 click.

In [ ]:
# Cell 1: Environment Setup & Clone Repository
!git clone -b mahdi https://github.com/Abolfazl-Afkhami/Hamsokhan.git
%cd Hamsokhan
!pip install -q transformers datasets torchaudio vocos soundfile librosa safetensors

In [ ]:
# Cell 2: Authentication with Hugging Face (Required for some restricted datasets)
import os
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    print("✓ Hugging Face Hub Authenticated.")
except Exception as e:
    print("ℹ Public datasets can be processed without HF_TOKEN.")

In [ ]:
# Cell 3: Process Layer 1 Synthesis Data (Thomcles 24kHz)
# Set --part_idx and --num_parts if splitting across multiple runs (e.g. 4 parts of 15GB each)
!python scripts/kaggle_dataset_builder.py \
    --dataset_type thomcles \
    --output_dir /kaggle/working/hamsokhan_shards \
    --part_idx 0 \
    --num_parts 1 \
    --max_hours 10.0 \
    --shard_size 500

In [ ]:
# Cell 4: Process Layer 2 Turn-Taking Data (PersianVox)
!python scripts/kaggle_dataset_builder.py \
    --dataset_type persianvox \
    --output_dir /kaggle/working/hamsokhan_shards \
    --part_idx 0 \
    --num_parts 1 \
    --max_hours 5.0 \
    --shard_size 500

In [ ]:
# Cell 5: Verify Generated Shards & Storage Size
import os
from pathlib import Path

out_dir = Path("/kaggle/working/hamsokhan_shards")
shards = sorted(list(out_dir.glob("*.safetensors")))
total_bytes = sum(f.stat().st_size for f in shards)

print(f"✓ Total Shards Generated: {len(shards)}")
print(f"✓ Total Output Disk Usage: {total_bytes / (1024**3):.2f} GB (Strictly under Kaggle 20GB limit!)")
for s in shards[:5]:
    print(f"  - {s.name} ({s.stat().st_size / (1024**2):.1f} MB)")
if len(shards) > 5:
    print(f"  ... and {len(shards) - 5} more shards.")

print("\n=======================================================")
print("✅ All shards ready! To publish this as a Kaggle Dataset:")
print("1. Click 'Save Version' -> 'Quick Save' or 'Save & Run All'.")
print("2. Go to notebook Output panel -> Click 'Create Dataset'.")
print("3. Name it e.g. 'hamsokhan-shards-part1' and use it in your training notebook!")
print("=======================================================")